
# RAFT-UP: Two-slice end-to-end pipeline

This notebook wraps the four tutorial stages into one workflow:

1. **Input preparation**
   - Directly load two `.h5ad` files, **or**
   - Build AnnData objects from raw DLPFC/Visium folders and save prepared `.h5ad` files.
2. **SOMDE** spatially variable gene selection.
3. **SCAN-IT / DGI** gene-cost computation.
4. **Downsampling + sparse FSGW** alignment.
5. **Full-resolution KNN recovery**.
6. **Plots and optional evaluation** when ground-truth labels are available.

The intended simple use case is:

```python
slice_a = "sliceA.h5ad"
slice_b = "sliceB.h5ad"
output_dir = "./raftup_results"
```

Only the **User configuration** cell should normally need editing.


In [ ]:

# ============================================================
# Native thread settings
# IMPORTANT: keep this cell before NumPy / SciPy / PyTorch imports
# ============================================================
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


In [ ]:

# ============================================================
# Imports
# ============================================================
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import scipy as sp
import scipy.sparse as sp_sparse
from scipy.spatial import distance
import matplotlib.pyplot as plt

import torch
import scanpy as sc

from raftup import (
    _downsample,
    _fsgw_utils,
    _fsgw_utils_test_sparse,
    _gene_cost_somde,
    _metrics_two_gpr,
    _plot,
    _recoverfull_new_new_knn,
)

print("torch:", torch.__version__)
try:
    import torch_geometric
    print("torch_geometric:", torch_geometric.__version__)
except Exception as e:
    print("torch_geometric import failed:", e)

print("numpy:", np.__version__)
print("scipy:", sp.__version__)



## User configuration

### Option A — normal users: two `.h5ad` files

Set:

```python
INPUT_MODE = "h5ad"
SLICE_A_H5AD = "sliceA.h5ad"
SLICE_B_H5AD = "sliceB.h5ad"
```

Each AnnData object must contain 2D spatial coordinates in:

```python
adata.obsm["spatial"]
```

### Option B — DLPFC tutorial/raw Visium data

Set:

```python
INPUT_MODE = "dlpfc_raw"
```

and point `DLPFC_ROOT` to the folder containing section folders such as `151508/` and `151509/`.

The notebook will construct AnnData objects, attach `original_clusters` when the GT file exists, and save prepared `.h5ad` files under the output folder.


In [ ]:

# ============================================================
# User configuration
# ============================================================

# Choose:
#   "h5ad"      -> use two already-prepared .h5ad files
#   "dlpfc_raw" -> build AnnData from DLPFC/Visium folders first
INPUT_MODE = "dlpfc_raw"

# ---------- Mode A: generic .h5ad ----------
SLICE_A_H5AD = "sliceA.h5ad"
SLICE_B_H5AD = "sliceB.h5ad"

# ---------- Mode B: DLPFC raw Visium ----------
DLPFC_ROOT = Path("./data")
SECTION_A = "151508"
SECTION_B = "151509"

# ---------- Output ----------
OUTPUT_DIR = Path("./raftup_results")

# ---------- Reproducibility ----------
MODE = "reproducible"  # "reproducible" or "fast"
SEED = 0

# ---------- SOMDE ----------
SOM_K = 5
SOMDE_MIN_CELLS = 50
SOMDE_MIN_COUNTS = 50
N_TOP_SV_GENES = 3000

# ---------- Gene-cost / DGI ----------
N_H = 100
N_EPOCH = 3500
LR = 2e-4
PRINT_STEP = 500

# ---------- Downsampling + sparse FSGW ----------
# These reproduce the current DLPFC tutorial.
# For other datasets, these distance thresholds may need adjustment
# according to the spatial coordinate scale.
DOWNSAMPLE_DISTANCE = 1020
DS_GW_CUTOFF = 206
DS_FEATURE_CUTOFF = 0.2

# ---------- Full recovery ----------
RF_GW_CUTOFF = 206
RF_FEATURE_CUTOFF = 0.4
K1 = 10
K2 = 10
RECOVERY_EPS = 0.01

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "prepared_input").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "somde").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "gene_cost_matrix").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "align_data").mkdir(parents=True, exist_ok=True)


In [ ]:

# ============================================================
# Helper functions: environment + input preparation
# ============================================================

def setup_environment(mode="reproducible", seed=0):
    if mode not in {"reproducible", "fast"}:
        raise ValueError("mode must be 'reproducible' or 'fast'")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if mode == "reproducible":
        torch.set_num_threads(1)
        try:
            torch.set_num_interop_threads(1)
        except RuntimeError:
            pass
        torch.use_deterministic_algorithms(True)
        device = torch.device("cpu")
    else:
        torch.use_deterministic_algorithms(False)

        # Stable default: CUDA when available; otherwise CPU.
        # MPS is intentionally not selected automatically because the
        # verified tutorial was unstable with some MPS/PyTorch combinations.
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            device = torch.device("cuda")
            torch.backends.cudnn.benchmark = True
            torch.backends.cudnn.deterministic = False
        else:
            device = torch.device("cpu")

    print("Mode:", mode)
    print("Device:", device)
    print("Torch threads:", torch.get_num_threads())
    return device


def validate_adata(adata, name="slice"):
    adata.var_names_make_unique()

    if "spatial" not in adata.obsm:
        raise ValueError(
            f"{name} does not contain adata.obsm['spatial']. "
            "RAFT-UP requires 2D spatial coordinates."
        )

    xy = np.asarray(adata.obsm["spatial"])
    if xy.ndim != 2 or xy.shape[1] < 2:
        raise ValueError(
            f"{name}: adata.obsm['spatial'] must have shape (n_spots, >=2); "
            f"received {xy.shape}."
        )

    if not np.all(np.isfinite(xy[:, :2])):
        raise ValueError(f"{name}: spatial coordinates contain NaN/Inf.")

    adata.obsm["spatial"] = np.asarray(xy[:, :2], dtype=float)
    return adata


def load_h5ad(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    adata = sc.read_h5ad(path)
    adata = validate_adata(adata, path.name)
    print(f"Loaded {path.name}: {adata.n_obs} spots x {adata.n_vars} genes")
    return adata


def build_dlpfc_anndata(root_dir, section_id, save_dir=None):
    """
    Reproduce the DLPFC tutorial's raw-data -> AnnData preparation.

    Expected layout:
        root_dir/
            151508/
                151508_filtered_feature_bc_matrix.h5
                spatial/...
                gt/tissue_positions_list_GTs.txt

    The GT file is optional for RAFT-UP itself. If present, it is attached as
    adata.obs['original_clusters'] and can later be used for tutorial evaluation.
    """
    root_dir = Path(root_dir)
    section_dir = root_dir / section_id

    count_file = f"{section_id}_filtered_feature_bc_matrix.h5"

    print(f"Building AnnData for DLPFC section {section_id} ...")

    adata = sc.read_visium(
        path=section_dir,
        count_file=count_file,
    )
    adata.var_names_make_unique()

    gt_path = section_dir / "gt" / "tissue_positions_list_GTs.txt"

    if gt_path.exists():
        gt_df = pd.read_csv(
            gt_path,
            sep=",",
            header=None,
            index_col=0,
        )

        # Match the original tutorial convention.
        adata.obs["original_clusters"] = gt_df.loc[:, 6]

        keep_bcs = adata.obs["original_clusters"].dropna().index
        adata = adata[keep_bcs].copy()

        adata.obs["original_clusters"] = (
            adata.obs["original_clusters"]
            .astype(int)
            .astype(str)
        )

        print("Attached original_clusters from GT file.")
    else:
        print("GT file not found; continuing without original_clusters.")

    adata = validate_adata(adata, section_id)

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        out_path = save_dir / f"{section_id}.h5ad"
        adata.write_h5ad(out_path)
        print("Prepared h5ad saved to:", out_path)

    print(f"{section_id}: {adata.n_obs} spots x {adata.n_vars} genes")
    return adata


def plot_spatial_pair(sliceA, sliceB, title_a="Slice A", title_b="Slice B"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    for ax, adata, title in zip(
        axes,
        [sliceA, sliceB],
        [title_a, title_b],
    ):
        xy = np.asarray(adata.obsm["spatial"])
        ax.scatter(xy[:, 0], xy[:, 1], s=8, alpha=0.8)
        ax.set_title(f"{title}\n{adata.n_obs} spots")
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.set_xlabel("spatial x")
        ax.set_ylabel("spatial y")

    plt.tight_layout()
    plt.show()


## Input preparation and QC

In [ ]:

# ============================================================
# Prepare / load the two AnnData objects
# ============================================================

device = setup_environment(MODE, SEED)

if INPUT_MODE == "h5ad":
    sliceA = load_h5ad(SLICE_A_H5AD)
    sliceB = load_h5ad(SLICE_B_H5AD)

    ID_A = Path(SLICE_A_H5AD).stem
    ID_B = Path(SLICE_B_H5AD).stem

elif INPUT_MODE == "dlpfc_raw":
    sliceA = build_dlpfc_anndata(
        DLPFC_ROOT,
        SECTION_A,
        save_dir=OUTPUT_DIR / "prepared_input",
    )
    sliceB = build_dlpfc_anndata(
        DLPFC_ROOT,
        SECTION_B,
        save_dir=OUTPUT_DIR / "prepared_input",
    )

    ID_A = SECTION_A
    ID_B = SECTION_B

else:
    raise ValueError("INPUT_MODE must be 'h5ad' or 'dlpfc_raw'")

PAIR = f"{ID_A}_{ID_B}"

print("\nPair:", PAIR)
print("A:", sliceA)
print("B:", sliceB)

plot_spatial_pair(sliceA, sliceB, ID_A, ID_B)



## Step 0 — SOMDE spatially variable genes

The original SOMDE package expects several NumPy-style aliases under the SciPy namespace.  
The compatibility patch below reproduces the working tutorial environment.


In [ ]:

# ============================================================
# Step 0: SOMDE
# ============================================================

# SOMDE 0.1.8 compatibility patch
_np_aliases = [
    "array", "arange", "argsort", "asarray",
    "zeros", "ones", "empty",
    "zeros_like", "ones_like", "full", "full_like",
    "log", "exp", "sqrt",
    "sum", "mean", "std", "var",
    "where", "isfinite", "isnan",
    "maximum", "minimum", "clip",
    "cumsum", "cumprod",
    "floor", "ceil", "abs",
]

for _name in _np_aliases:
    if not hasattr(sp, _name):
        setattr(sp, _name, getattr(np, _name))

from somde import SomNode


def compute_and_save_somde_csv(
    adata,
    section_id,
    output_dir,
    som_k=5,
    min_cells=50,
    min_counts=50,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    adata_raw = adata.copy()

    X_raw = adata_raw.X
    gene_names_raw = np.asarray(adata_raw.var_names).astype(str)

    if sp_sparse.issparse(X_raw):
        total_count_raw = np.asarray(X_raw.sum(axis=0)).ravel()
        nonzero_spots_raw = np.asarray((X_raw > 0).sum(axis=0)).ravel()
    else:
        X_tmp = np.asarray(X_raw)
        total_count_raw = X_tmp.sum(axis=0)
        nonzero_spots_raw = (X_tmp > 0).sum(axis=0)

    adata_filt = adata.copy()
    n_before = adata_filt.n_vars

    sc.pp.filter_genes(adata_filt, min_cells=min_cells)
    n_after_cells = adata_filt.n_vars

    sc.pp.filter_genes(adata_filt, min_counts=min_counts)
    n_after_counts = adata_filt.n_vars

    if adata_filt.n_vars == 0:
        raise ValueError(
            f"SOMDE filtering removed all genes for {section_id}. "
            "Lower SOMDE_MIN_CELLS / SOMDE_MIN_COUNTS."
        )

    kept_genes = set(adata_filt.var_names.astype(str))
    keep_mask = np.asarray([g in kept_genes for g in gene_names_raw])

    filter_df = pd.DataFrame({
        "g": gene_names_raw,
        "total_count": total_count_raw,
        "nonzero_spots": nonzero_spots_raw,
        "keep_for_somde": keep_mask,
    })

    filter_df.to_csv(
        output_dir / f"somde_{section_id}_filter_stats.csv",
        index=False,
    )

    filter_df.loc[~filter_df["keep_for_somde"]].to_csv(
        output_dir / f"somde_{section_id}_filtered_out_genes.csv",
        index=False,
    )

    pts = np.asarray(adata_filt.obsm["spatial"], dtype=np.float32)

    X_filt = (
        adata_filt.X.toarray()
        if sp_sparse.issparse(adata_filt.X)
        else np.asarray(adata_filt.X)
    )

    df_expr = pd.DataFrame(
        X_filt,
        columns=adata_filt.var_names.astype(str),
    )

    print(
        f"[SOMDE] {section_id}: "
        f"{n_before} -> {n_after_cells} -> {n_after_counts} genes"
    )

    som = SomNode(pts, som_k)
    som.mtx(df_expr.T)
    som.norm()
    result, _ = som.run()

    result_path = output_dir / f"somde_{section_id}.csv"
    result.to_csv(result_path, index=False)

    print("Saved:", result_path)
    return result


somde_A = compute_and_save_somde_csv(
    sliceA,
    ID_A,
    OUTPUT_DIR / "somde",
    som_k=SOM_K,
    min_cells=SOMDE_MIN_CELLS,
    min_counts=SOMDE_MIN_COUNTS,
)

somde_B = compute_and_save_somde_csv(
    sliceB,
    ID_B,
    OUTPUT_DIR / "somde",
    som_k=SOM_K,
    min_cells=SOMDE_MIN_CELLS,
    min_counts=SOMDE_MIN_COUNTS,
)

display(somde_A.head())
display(somde_B.head())


In [ ]:

# Plot the strongest SOMDE scores for a quick QC view
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, df, sid in [
    (axes[0], somde_A, ID_A),
    (axes[1], somde_B, ID_B),
]:
    plot_df = df.head(30).copy()

    # SOMDE versions may expose different score-column names.
    numeric_cols = [
        c for c in plot_df.columns
        if c != "g" and pd.api.types.is_numeric_dtype(plot_df[c])
    ]

    if numeric_cols:
        score_col = numeric_cols[0]
        ax.barh(
            plot_df["g"].astype(str)[::-1],
            plot_df[score_col][::-1],
        )
        ax.set_title(f"{sid}: top SOMDE genes")
        ax.set_xlabel(score_col)
    else:
        ax.text(0.5, 0.5, "No numeric SOMDE score found", ha="center")
        ax.set_title(sid)

plt.tight_layout()
plt.show()


## Step 1 — Gene-cost matrix from SOMDE + SCAN-IT / DGI

In [ ]:

# ============================================================
# Step 1: gene cost
# ============================================================

geneA = sliceA.copy()
geneB = sliceB.copy()

for sl in (geneA, geneB):
    sc.pp.normalize_total(sl)
    sc.pp.log1p(sl)

sv_genes_A = somde_A["g"].astype(str).values[:N_TOP_SV_GENES]
sv_genes_B = somde_B["g"].astype(str).values[:N_TOP_SV_GENES]

sv_genes_A = [g for g in sv_genes_A if g in geneA.var_names]
sv_genes_B = [g for g in sv_genes_B if g in geneB.var_names]

print(ID_A, "valid SV genes:", len(sv_genes_A))
print(ID_B, "valid SV genes:", len(sv_genes_B))

geneA = geneA[:, sv_genes_A].copy()
geneB = geneB[:, sv_genes_B].copy()

for sl in (geneA, geneB):
    sc.pp.scale(sl)

_gene_cost_somde.compute_gene_cost(
    sliceA=geneA,
    sliceB=geneB,
    section_id_A=ID_A,
    section_id_B=ID_B,
    n_h=N_H,
    n_epoch=N_EPOCH,
    lr=LR,
    print_step=PRINT_STEP,
    seed=SEED,
    device=device,
    output_dir=str(OUTPUT_DIR / "gene_cost_matrix"),
)

PT_PATH = OUTPUT_DIR / "gene_cost_matrix" / f"{PAIR}_somde_cost_matrix.pt"
C = torch.load(PT_PATH, map_location="cpu")
M_full = C.detach().cpu().numpy()

print("Gene-cost matrix:", M_full.shape)
print("min / median / max:",
      float(np.min(M_full)),
      float(np.median(M_full)),
      float(np.max(M_full)))


In [ ]:

# Gene-cost heatmap (downsampled display only; full matrix is unchanged)
max_display = 500
r = np.linspace(0, M_full.shape[0] - 1, min(max_display, M_full.shape[0])).astype(int)
c = np.linspace(0, M_full.shape[1] - 1, min(max_display, M_full.shape[1])).astype(int)

plt.figure(figsize=(7, 6))
plt.imshow(M_full[np.ix_(r, c)], aspect="auto")
plt.title(f"Gene-cost matrix: {PAIR}")
plt.xlabel(ID_B)
plt.ylabel(ID_A)
plt.colorbar(label="cost")
plt.tight_layout()
plt.show()


## Step 2a — Downsampling + sparse FSGW

In [ ]:

# ============================================================
# Step 2a: downsampling
# ============================================================
np.random.seed(SEED)

downsampled_sliceA, indices_dsa = _downsample.downsample_slice(
    sliceA,
    DOWNSAMPLE_DISTANCE,
)

downsampled_sliceB, indices_dsb = _downsample.downsample_slice(
    sliceB,
    DOWNSAMPLE_DISTANCE,
)

indices_dsa = np.asarray(indices_dsa, dtype=int)
indices_dsb = np.asarray(indices_dsb, dtype=int)

np.save(
    OUTPUT_DIR / "align_data" / f"{PAIR}_downsample_indices_dsa.npy",
    indices_dsa,
)
np.save(
    OUTPUT_DIR / "align_data" / f"{PAIR}_downsample_indices_dsb.npy",
    indices_dsb,
)

print(
    f"{ID_A}: {sliceA.n_obs} -> {downsampled_sliceA.n_obs} spots"
)
print(
    f"{ID_B}: {sliceB.n_obs} -> {downsampled_sliceB.n_obs} spots"
)

# Keep the original tutorial visual QC.
_downsample.visualize_downsampled_points(sliceA, downsampled_sliceA)
_downsample.visualize_downsampled_points(sliceB, downsampled_sliceB)


In [ ]:

# Build the downsampled feature-cost matrix directly in memory.
M = M_full[np.ix_(indices_dsa, indices_dsb)]

M_max = float(np.max(M))
if M_max <= 0:
    raise ValueError("Downsampled feature-cost matrix has no positive values.")

M = M / M_max

D_A = distance.cdist(
    downsampled_sliceA.obsm["spatial"],
    downsampled_sliceA.obsm["spatial"],
)
D_B = distance.cdist(
    downsampled_sliceB.obsm["spatial"],
    downsampled_sliceB.obsm["spatial"],
)

positive_A = D_A[D_A > 0]
positive_B = D_B[D_B > 0]

print("Minimum nonzero spatial distance A:", np.min(positive_A))
print("Minimum nonzero spatial distance B:", np.min(positive_B))
print("Downsampled feature cost shape:", M.shape)


In [ ]:

def compute_P_support_metrics(P, support_tol=0.0, print_result=True):
    if hasattr(P, "detach"):
        P_array = P.detach().cpu().numpy()
    elif hasattr(P, "toarray"):
        P_array = P.toarray()
    else:
        P_array = np.asarray(P)

    support = np.abs(P_array) > support_tol

    m_A = support.sum(axis=1).astype(int)
    m_B = support.sum(axis=0).astype(int)

    covered_A = m_A > 0
    covered_B = m_B > 0

    cov_A = float(covered_A.mean()) if m_A.size else np.nan
    cov_B = float(covered_B.mean()) if m_B.size else np.nan

    metrics = {
        "shape": P_array.shape,
        "transported_mass": float(P_array.sum()),
        "n_retained_pairs": int(support.sum()),
        "support_density": float(support.mean()),
        "CovA": cov_A,
        "CovB": cov_B,
        "CovMin": float(min(cov_A, cov_B)),
        "CovGap": float(abs(cov_A - cov_B)),
    }

    if print_result:
        for k, v in metrics.items():
            print(f"{k}: {v}")

    return metrics


In [ ]:

# Sparse FSGW on downsampled slices
P_ds, log_heap = _fsgw_utils_test_sparse.fsgw_mvc_exp_sparse_heap(
    D_A,
    D_B,
    M,
    gw_cutoff=DS_GW_CUTOFF,
    w_cutoff=DS_FEATURE_CUTOFF,
    seed=SEED,
    return_log=True,
    show_progress=True,
)

ds_metrics = compute_P_support_metrics(P_ds)

DS_MATCH_PATH = (
    OUTPUT_DIR
    / "align_data"
    / f"ds_matching_{PAIR}_{DS_GW_CUTOFF}_{DS_FEATURE_CUTOFF}_turbo.npy"
)
np.save(DS_MATCH_PATH, P_ds)

print("Saved:", DS_MATCH_PATH)
print("\nStage times:")
for k, v in log_heap["stage_times"].items():
    print(f"  {k}: {v:.4f} sec")


## Step 2b — Harden downsampled matching + full-resolution recovery

In [ ]:

# ============================================================
# Harden the downsampled matching
# ============================================================
P_hard = np.zeros_like(P_ds)

row_max_idx = P_ds.argmax(axis=1)
P_hard[
    np.arange(P_ds.shape[0]),
    row_max_idx,
] = P_ds[
    np.arange(P_ds.shape[0]),
    row_max_idx,
]

print("Downsampled transported mass:", P_ds.sum())
print(
    "Covered rows after hardening:",
    np.mean(np.any(P_hard != 0, axis=1)),
)

col_nnz = np.count_nonzero(P_hard, axis=0)
print(
    "Columns with >=2 nonzeros:",
    np.sum(col_nnz >= 2),
)

# Tutorial 3D visualization of downsampled matching
_plot.plot_3d(
    sliceA.obsm["spatial"],
    sliceB.obsm["spatial"],
    P_hard,
    indices_dsa,
    indices_dsb,
    linewidth=0.5,
)


In [ ]:

# ============================================================
# Full-resolution recovery
# ============================================================
P_full_sparse = _recoverfull_new_new_knn.recover_full_mapping_knn(
    M=M_full,
    X1=sliceA.obsm["spatial"],
    X2=sliceB.obsm["spatial"],
    P=P_hard,
    idx1=indices_dsa,
    idx2=indices_dsb,
    k1=K1,
    k2=K2,
    thresh_CGW=RF_GW_CUTOFF,
    thresh_CCC=RF_FEATURE_CUTOFF,
    eps=RECOVERY_EPS,
)

P_full = (
    P_full_sparse.toarray()
    if hasattr(P_full_sparse, "toarray")
    else np.asarray(P_full_sparse)
)

FULL_MATCH_PATH = (
    OUTPUT_DIR
    / "align_data"
    / f"full_matching_{PAIR}_{RF_GW_CUTOFF}_{RF_FEATURE_CUTOFF}.npy"
)

np.save(FULL_MATCH_PATH, P_full)

print("Full matching shape:", P_full.shape)
print("Full transported mass:", P_full.sum())
print("Saved:", FULL_MATCH_PATH)


In [ ]:

# Visualize the strongest full-resolution matches in 2D
# (for readability, only the strongest links are plotted).
def plot_matching_links(
    sliceA,
    sliceB,
    P,
    max_links=500,
):
    XA = np.asarray(sliceA.obsm["spatial"])
    XB = np.asarray(sliceB.obsm["spatial"])

    # Put B to the right for a clean side-by-side visualization.
    shift = XA[:, 0].max() - XB[:, 0].min()
    gap = 0.15 * max(
        XA[:, 0].ptp() if hasattr(XA[:, 0], "ptp") else np.ptp(XA[:, 0]),
        XB[:, 0].ptp() if hasattr(XB[:, 0], "ptp") else np.ptp(XB[:, 0]),
    )
    XB_plot = XB.copy()
    XB_plot[:, 0] += shift + gap

    rows, cols = np.nonzero(P)
    weights = P[rows, cols]

    if len(weights) > max_links:
        keep = np.argsort(weights)[-max_links:]
        rows = rows[keep]
        cols = cols[keep]
        weights = weights[keep]

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.scatter(XA[:, 0], XA[:, 1], s=6, label=ID_A)
    ax.scatter(XB_plot[:, 0], XB_plot[:, 1], s=6, label=ID_B)

    for i, j in zip(rows, cols):
        ax.plot(
            [XA[i, 0], XB_plot[j, 0]],
            [XA[i, 1], XB_plot[j, 1]],
            linewidth=0.3,
            alpha=0.25,
        )

    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_title(f"RAFT-UP full matching: {PAIR} (top {len(rows)} links)")
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_matching_links(sliceA, sliceB, P_full)



## Optional DLPFC / benchmark evaluation

RAFT-UP does **not** require ground-truth labels.

If both input AnnData objects contain `obs["original_clusters"]`, the notebook additionally computes the same LAA/GPR diagnostics used in the DLPFC tutorial. For normal biological datasets without known labels, this cell simply skips evaluation.


In [ ]:

# ============================================================
# Optional evaluation
# ============================================================
if (
    "original_clusters" in sliceA.obs.columns
    and "original_clusters" in sliceB.obs.columns
):
    labels_full = np.concatenate([
        np.asarray(sliceA.obs["original_clusters"]),
        np.asarray(sliceB.obs["original_clusters"]),
    ])

    laa = _metrics_two_gpr.cal_layer_based_alignment_result_full_skip_all_zero(
        P_full,
        labels_full,
    )

    gpr_300 = _metrics_two_gpr.GPR_original(
        P_full,
        sliceA.obsm["spatial"],
        sliceB.obsm["spatial"],
        dis_cut=300,
        P_cut=1e-19,
    )

    gpr_600 = _metrics_two_gpr.GPR_original(
        P_full,
        sliceA.obsm["spatial"],
        sliceB.obsm["spatial"],
        dis_cut=600,
        P_cut=1e-19,
    )

    gpr_900 = _metrics_two_gpr.GPR_original(
        P_full,
        sliceA.obsm["spatial"],
        sliceB.obsm["spatial"],
        dis_cut=900,
        P_cut=1e-19,
    )

    print("LAA:", laa[0])
    print("GPR 300:", gpr_300)
    print("GPR 600:", gpr_600)
    print("GPR 900:", gpr_900)

else:
    print(
        "No original_clusters found in both slices. "
        "Skipping benchmark-only LAA/GPR evaluation."
    )


In [ ]:

# ============================================================
# Final summary
# ============================================================
summary = {
    "pair": PAIR,
    "input_mode": INPUT_MODE,
    "n_spots_A": int(sliceA.n_obs),
    "n_spots_B": int(sliceB.n_obs),
    "n_downsample_A": int(downsampled_sliceA.n_obs),
    "n_downsample_B": int(downsampled_sliceB.n_obs),
    "downsample_transported_mass": float(P_ds.sum()),
    "full_transported_mass": float(P_full.sum()),
    "downsample_matching": str(DS_MATCH_PATH),
    "full_matching": str(FULL_MATCH_PATH),
    "gene_cost_matrix": str(PT_PATH),
}

SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nRAFT-UP pipeline completed successfully.")
